# 1001 · 02 · New alarm files arrive (part B)

The turbines also report **alarms**. The older parks send them as **CSV**, the newer ones as **JSON**, and both land in the same folder: `landing/data/alarms/`. Each run of this notebook (**Run all**) drops **30 new files, 15 CSV and 15 JSON, all mixed together**.

In [ ]:
%python
import os
import shutil

CATALOG = "lab1001"

# This notebook's folder is the lab folder inside your Git folder.
LAB_PATH = os.getcwd()
if not LAB_PATH.startswith("/Workspace"):
    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    LAB_PATH = "/Workspace" + os.path.dirname(ctx.notebookPath().get())

print("Lab:    ", LAB_PATH)
print("Catalog:", CATALOG)

In [ ]:
%python
import csv
import json
import random
from datetime import date, datetime, timedelta

FILES_PER_FORMAT = 15
target = f"/Volumes/{CATALOG}/landing/data/alarms"
os.makedirs(target, exist_ok=True)

PARKS = ["P01", "P02", "P03"]
# (alarm code, severity, typical stop in minutes)
ALARMS = [("GRID_LOSS", "stop", 45), ("HIGH_WIND", "stop", 90), ("PITCH_FAULT", "stop", 120),
          ("YAW_MISALIGNMENT", "warning", 15), ("GEARBOX_OIL_TEMP", "warning", 30),
          ("ICE_DETECTED", "stop", 180)]
FIELDS = ["event_time", "park_id", "turbine_id", "alarm_code", "severity", "duration_min"]

# The alarms belong to the last production day in the volume
reports = [f.name for f in dbutils.fs.ls(f"/Volumes/{CATALOG}/landing/data") if f.name.startswith("production_")]
day = max(date.fromisoformat(n[len("production_"):-len(".csv")]) for n in reports)

rnd = random.Random()
batch = datetime.now().strftime("%H%M%S")
formats = ["csv"] * FILES_PER_FORMAT + ["json"] * FILES_PER_FORMAT
rnd.shuffle(formats)

for i, fmt in enumerate(formats, start=1):
    park = rnd.choice(PARKS)
    rows = []
    for _ in range(rnd.randint(2, 6)):
        code, severity, minutes = rnd.choice(ALARMS)
        at = datetime.combine(day, datetime.min.time()) + timedelta(minutes=rnd.randint(0, 1439))
        rows.append({"event_time": at.isoformat(sep=" "), "park_id": park,
                     "turbine_id": f"{park}-WTG{rnd.randint(1, 12):02d}", "alarm_code": code,
                     "severity": severity, "duration_min": max(1, int(rnd.gauss(minutes, minutes / 3)))})
    path = f"{target}/alarms_{park}_{day:%Y%m%d}_{batch}_{i:02d}.{fmt}"
    with open(path, "w", newline="") as f:
        if fmt == "csv":
            w = csv.DictWriter(f, fieldnames=FIELDS)
            w.writeheader()
            w.writerows(rows)
        else:  # JSON Lines: one alarm per line
            f.writelines(json.dumps(r) + "\n" for r in rows)

print(f"{len(formats)} new files for {day}, batch {batch}")
display(dbutils.fs.ls(target))

Both formats share the folder. The job's new task `ingest_alarms` runs `src/02_ingest_alarms` with the parameter `file_format = json`: it picks only the JSON files and leaves the CSV ones for another value of the same parameter.